## I will be using the same pipeline as Ro, but I will be loading another model.
- TheBloke. (2023). Mistral-7B-Instruct-v0.1-GGUF [Model]. Hugging Face. https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF

In [13]:
import pandas as pd

from langchain_community.llms import CTransformers
from langchain.schema import Document

from langchain.text_splitter import RecursiveCharacterTextSplitter
#!pip install -U langchain-huggingface
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA

#!pip install evaluate
from evaluate import load
#!pip install rouge_score
#!pip install bert_score
import pandas as pd
from tqdm import tqdm

import difflib
import numpy as np
from sklearn.metrics import average_precision_score
from tqdm import tqdm

from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, RetrievalQA
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

### Loading and preparing the vector base and finalpassage

In [2]:
# === STEP 1: load CSV ===
df_train = pd.read_csv("data/train.csv")  
df_train = df_train.dropna(subset=["finalpassage","query","answers"])

df_val = pd.read_csv("data/valid.csv")  
df_val = df_val.dropna(subset=["finalpassage","query","answers"])

df_all = pd.concat([df_train,df_val],ignore_index =True)
df_all["finalpassage"] = df_all["finalpassage"].astype(str)

# === Step 2: Convert 'finalpassage' to docs ===
docs = [
    Document(
        page_content=row["finalpassage"]
    )
    for _, row in df_all.iterrows()
]

# === STEP 3: split docs in chunks ===
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)

split_docs[:1]

# === STEP 4: Generate embeddings ===
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# === STEP 5: Create and save the vector database (FAISS) ===
vector_db = FAISS.from_documents(split_docs, embeddings)
vector_db.save_local("vectorial_faiss_db")

/home/leandro-sartini/miniconda3/envs/work/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
2025-04-02 22:33:21.224595: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-02 22:33:21.294295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743647601.318440   44964 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has a

### Loading Llama

In [3]:
from langchain.llms import CTransformers

llm = CTransformers(
    model="models/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    model_type="llama",
    config={
        "max_new_tokens": 512,
        "temperature": 0.6,
        "context_length": 2048,
        "gpu_layers": 33,  # Max layers to offload to GPU (auto-manages based on VRAM)
    }
)


In [4]:
retriever = vector_db.as_retriever(search_kwargs={"k": 3})  # Retrieve top 3 chunks

In [5]:
def rag_query(llm, retriever, query: str) -> str:
    docs = retriever.get_relevant_documents(query)
    context = "\n".join(doc.page_content for doc in docs)

    prompt = f"""You are a helpful assistant. Use the context below to answer the user's question.

Context:
{context}

Question:
{query}

Answer:"""

    return llm(prompt)


In [6]:
# === STEP 7: Create RAG chain (Retriever + LLM) ===
retriever = vector_db.as_retriever(search_kwargs={"k": 2})
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, chain_type="stuff")  # we can play with the chain_type later using "refine" or "map_rerank"

In [7]:
# === STEP 8: test a query ===
query = "definition of what the skeletal"
output = qa_chain.invoke(query)
answer = output["result"]
print(f"Question: {query}\nAnswer: {answer}")

Question: definition of what the skeletal
Answer:  The skeletal system includes all of the bones and joints in the body. Each bone is a complex living organ that is made up of many cells, protein fibers, and minerals. The skeleton acts as a scaffold by providing support and protection for the soft tissues that make up the rest of the body. The skeletal system also provides attachment points for muscles to allow movements at the joints. New blood cells are produced by the red bone marrow inside of our bones.


## Let's check which other documents it brings

In [8]:
docs = retriever.get_relevant_documents(query)

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)


--- Document 1 ---
The skeletal system includes all of the bones and joints in the body. Each bone is a complex living organ that is made up of many cells, protein fibers, and minerals. The skeleton acts as a scaffold by providing support and protection for the soft tissues that make up the rest of the body. The skeletal system also provides attachment points for muscles to allow movements at the joints. New blood cells are produced by the red bone marrow inside of our bones.The axial skeleton is the part of the

--- Document 2 ---
The skeletal system includes all of the bones and joints in the body. Each bone is a complex living organ that is made up of many cells, protein fibers, and minerals. The skeleton acts as a scaffold by providing support and protection for the soft tissues that make up the rest of the body.The skeletal system supports and protects the body while giving it shape and form. This system is composed of connective tissues including bone, cartilage, tendons, and li

/tmp/ipykernel_44964/4275237952.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


### In this case it's just bringing the same document

# Validation 

### ROUGE-L and BERT-F1

In [18]:
# load and sample validation dataset
df_val = pd.read_csv("data/valid.csv")  
df_val = df_val.dropna(subset=["finalpassage","query","answers"])

df_val = df_val.sample(frac=0.005, random_state=42).reset_index(drop=True)
df_val = df_val[:10]
#df_val["query"] = df_val["query"].astype(str)
#df_val["answers"] = df_val["answers"].astype(str)

# Metrics
rouge = load("rouge")
bert_score = load("bertscore")

predictions = []
expected_responses = []

for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Procesing query's"):
    question = row["query"]
    real_answer = row["answers"]

    try:
        generated_response = qa_chain.invoke({"query": question})
    except Exception as e:
        print(f"Error with the question: {question} → {e}")
        generated_response = ""

    predictions.append(generated_response)
    expected_responses.append(real_answer)



Procesing query's: 100%|████████████████████████████████████████████████████████████████| 10/10 [01:19<00:00,  7.98s/it]


In [19]:
# Faltten predictions
predictions_text = [p["result"] if isinstance(p, dict) and "result" in p else str(p) for p in predictions]
# Ensure responses are strings
expected_responses_texto = [str(r) for r in expected_responses]

In [20]:
# ROUGE-L
rouge_result = rouge.compute(predictions=predictions_text, references=expected_responses_texto, rouge_types=["rougeL"])
print(f"\nROUGE-L F1 Score: {rouge_result['rougeL']:.4f}")

# BERT-F1 
bert_result = bert_score.compute(
    predictions=predictions_text,
    references=expected_responses_texto,
    lang="en",
    device="cuda"  
)
print(f" BERTScore F1 (average): {sum(bert_result['f1']) / len(bert_result['f1']):.4f}")


ROUGE-L F1 Score: 0.3879


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

 BERTScore F1 (average): 0.8849


- BERTScore F1 (Semantic Similarity (even if different words are used)): 0.89
- ROUGE-L: (Literal Overlap (exact words or phrases)) = 0.39
    - Compared to the run of Llama v2 it's getting a doubled better Rouge and a little better BERTscore.

### MAP and MRR with k=2

In [21]:
df_val["finalpassage"] = df_val["finalpassage"].astype(str)

k = 2  # retrieved documents 

def is_relevant(retrieved_passages, gold_passage, threshold=0.85):
    """
   Compare each retrieved documents against expected document with fuzzy match.
    """
    for passage in retrieved_passages:
        ratio = difflib.SequenceMatcher(None, passage, gold_passage).ratio()
        if ratio >= threshold:
            return True
    return False

reciprocal_ranks = []
average_precisions = []

for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Evaluating MAP/MRR"):
    query = row["query"]
    expected_passage = row["finalpassage"]

    try:
        retrieved_docs = retriever.get_relevant_documents(query)
        retrieved_passages = [doc.page_content for doc in retrieved_docs[:k]]
    except Exception as e:
        print(f"Error on query: {query} → {e}")
        retrieved_passages = []

    # === MRR ===
    rank = None
    for i, passage in enumerate(retrieved_passages):
        ratio = difflib.SequenceMatcher(None, passage, expected_passage).ratio()
        if ratio >= 0.6:
            rank = i + 1
            break
    if rank:
        reciprocal_ranks.append(1 / rank)
    else:
        reciprocal_ranks.append(0)

    # === MAP ===
    # create binary relevance tags (1 if similar, 0 if no)
    relevance_scores = [
        1 if difflib.SequenceMatcher(None, passage, expected_passage).ratio() >= 0.85 else 0
        for passage in retrieved_passages
    ]
    precision = average_precision_score(relevance_scores, [1]*len(relevance_scores)) if any(relevance_scores) else 0
    average_precisions.append(precision)

# === Final Results ===
map_score = np.mean(average_precisions)
mrr_score = np.mean(reciprocal_ranks)

print(f"\n🔹 MAP (Mean Average Precision) @ {k}: {map_score:.4f}")
print(f"🔸 MRR (Mean Reciprocal Rank) @ {k}: {mrr_score:.4f}")

Evaluating MAP/MRR: 100%|███████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 21.24it/s]


🔹 MAP (Mean Average Precision) @ 2: 0.1000
🔸 MRR (Mean Reciprocal Rank) @ 2: 0.6000


### In case of the RAG the score is pretty much the same.

### For Fine tuning I will be trying to load a different embedding which may help enhance MAP and MRR for retrieval

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/msmarco-distilbert-base-tas-b")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
# === STEP 5: Create and save the vector database (FAISS) ===
vector_db = FAISS.from_documents(split_docs, embeddings)
vector_db.save_local("vectorial_faiss_db")

In [24]:
# === STEP 7: Create RAG chain (Retriever + LLM) ===
retriever = vector_db.as_retriever(search_kwargs={"k": 5})
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, chain_type="stuff")  # we can play with the chain_type later using "refine" or "map_rerank"

In [25]:
# === STEP 8: test a query ===
query = "definition of what the skeletal"
output = qa_chain.invoke(query)
answer = output["result"]
print(f"Question: {query}\nAnswer: {answer}")

Question: definition of what the skeletal
Answer:  Definition of the Skeletal System: The Skeletal System is a group of organs that are directly or indirectly attached to bones, providing support and movement to the body. It includes the skull, vertebral column, ribs, pelvis, limbs, and facial bones.


## Here we can see that it improved a lot already. Not just returning the same doc

In [26]:
docs = retriever.get_relevant_documents(query)

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)


--- Document 1 ---
Skeletal muscle is striated muscle tissue that is attached to bones. It is composed of fibers that look like a mixture of dark and light bands bundled together that run along the bone. Skeletal muscle is striated muscle tissue that is attached to bones. Skeletal muscle attaches to the bones. At the macroscopic level, the skeletal muscles are composed of a variety of layers. Working out strengthens skeletal muscles, making them more visible.Striation Patterns. -There are two main parts of the

--- Document 2 ---
A voluntary muscle, often called a skeletal muscle, is one of three types of muscle in the body. Most voluntary muscles are used to move bones, though some, such as the muscles in the face, are used to create movements below the skin.There are cardiac, smooth, and skeletal. Both cardiac and smooth are involuntary. Skeletal is voluntary. Skeletal muscles are the muscles you use to move your bones. You do these things voluntarily, like raising your arm. Skeleta

### MAP and MRR with k=2

In [30]:
df_val["finalpassage"] = df_val["finalpassage"].astype(str)

k = 2  # retrieved documents 

def is_relevant(retrieved_passages, gold_passage, threshold=0.85):
    """
   Compare each retrieved documents against expected document with fuzzy match.
    """
    for passage in retrieved_passages:
        ratio = difflib.SequenceMatcher(None, passage, gold_passage).ratio()
        if ratio >= threshold:
            return True
    return False

reciprocal_ranks = []
average_precisions = []

for _, row in tqdm(df_val.iterrows(), total=len(df_val), desc="Evaluating MAP/MRR"):
    query = row["query"]
    expected_passage = row["finalpassage"]

    try:
        retrieved_docs = retriever.get_relevant_documents(query)
        retrieved_passages = [doc.page_content for doc in retrieved_docs[:k]]
    except Exception as e:
        print(f"Error on query: {query} → {e}")
        retrieved_passages = []

    # === MRR ===
    rank = None
    for i, passage in enumerate(retrieved_passages):
        ratio = difflib.SequenceMatcher(None, passage, expected_passage).ratio()
        if ratio >= 0.6:
            rank = i + 1
            break
    if rank:
        reciprocal_ranks.append(1 / rank)
    else:
        reciprocal_ranks.append(0)

    # === MAP ===
    # create binary relevance tags (1 if similar, 0 if no)
    relevance_scores = [
        1 if difflib.SequenceMatcher(None, passage, expected_passage).ratio() >= 0.85 else 0
        for passage in retrieved_passages
    ]
    precision = average_precision_score(relevance_scores, [1]*len(relevance_scores)) if any(relevance_scores) else 0
    average_precisions.append(precision)

# === Final Results ===
map_score = np.mean(average_precisions)
mrr_score = np.mean(reciprocal_ranks)

print(f"\n🔹 MAP (Mean Average Precision) @ {k}: {map_score:.4f}")
print(f"🔸 MRR (Mean Reciprocal Rank) @ {k}: {mrr_score:.4f}")

Evaluating MAP/MRR: 100%|███████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 18.10it/s]


🔹 MAP (Mean Average Precision) @ 2: 0.1500
🔸 MRR (Mean Reciprocal Rank) @ 2: 0.7000
